In [0]:
%sql
create database retail_project.retail_db

In [0]:
%sql
create volume retail_project.retail_db.retail_files

In [0]:
customers_df=spark.read.csv('/Volumes/retail_project/retail_db/retail_files/bronze/customers_150k.csv',header=True,inferSchema=True)
products_df=spark.read.csv('/Volumes/retail_project/retail_db/retail_files/bronze/products_120k.csv',header=True,inferSchema=True)
inventory_df=spark.read.csv('/Volumes/retail_project/retail_db/retail_files/bronze/inventory_200k.csv',header=True,inferSchema=True)
sales_df=spark.read.csv('/Volumes/retail_project/retail_db/retail_files/bronze/online_retail_sales_500k.csv',header=True,inferSchema=True)

In [0]:
sales_df.printSchema()
sales_df.count()
customers_df.printSchema()
customers_df.count()
products_df.printSchema()
products_df.count()
inventory_df.printSchema()
inventory_df.count()

SALES DATA CLEANING

In [0]:
sales_df.display()

In [0]:
from pyspark.sql.functions import *
#Removing invalid records
sales_clean=sales_df.filter(col('Quantity')>0)\
            .filter(col('CustomerID').isNotNull())
sales_clean.display()

In [0]:
##Removing duplicates
sales_clean=sales_clean.dropDuplicates(['InvoiceNo','StockCode','CustomerID'])
display(sales_clean)

In [0]:
sales_clean.select('InvoiceDate').display()

In [0]:
##Converting invoiceData to Proper TimeStamp

sales_clean = sales_clean.withColumn('InvoiceDate', to_timestamp(col('InvoiceDate'), "dd-MM-yyyy'T'HH:mm:ss.SSSXXX"))
sales_clean.select('InvoiceDate').display()


CUSTOME DATA CLEANING

In [0]:
customer_clean=customers_df.filter(col('age')>=18)\
    .withColumn('gender',when(col('gender').isNull(),'Unknown').otherwise(col('gender')))
customer_clean.display()

In [0]:
customer_clean=customer_clean.dropDuplicates(['customer_id'])
customer_clean.display()
customer_clean.count()

PRODUCT DATA CLEANING

In [0]:
products_clean=products_df.filter(col('price')>0)\
    .dropDuplicates(['product_id'])
products_clean.display()
products_clean.count()

INVENTORY DATA CLEANING

In [0]:
inventory_clean=inventory_df.filter(col('stock_qty')>0)
inventory_clean.display()

In [0]:
from pyspark.sql.functions import *
#BROADCAST JOIN (OPTIMIZATION)
customers_broadcast=broadcast(customer_clean)
products_broadcast=broadcast(products_clean)

In [0]:
from pyspark.sql.functions import broadcast, col

silver_sales_enriched = (
    sales_clean
    .filter(col("quantity") > 0)
    .join(broadcast(customer_clean), "customer_id", "left")
    .join(broadcast(products_clean), "product_id", "left")
)


REVENUE CALCULATION

In [0]:
silver_sales_enriched=silver_sales_enriched.withColumn('Total amount',expr('Quantity*Price'))

DISCOUNT

In [0]:
silver_sales_enriched = silver_sales_enriched.withColumn(
    'discount',
    expr("case when discount > 0 then 'Y' else 'N' end")
)

SALES CHANNEL


In [0]:
silver_sales_enriched=silver_sales_enriched\
    .withColumn('sales_channel', expr("upper(sales_channel)"))

In [0]:
silver_sales_enriched = (
    silver_sales_enriched
    .filter(col("price").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
)


In [0]:
final_df = sales_df.join(
    customers_df,
    on="customer_id",
    how="left"
)


In [0]:

from pyspark.sql.functions import col

sales_df = sales_df.withColumnRenamed("CustomerID", "customer_id")


In [0]:
final_df = sales_df.join(
    customers_df,
    sales_df.customer_id == customers_df.customer_id,
    "left"
)


In [0]:

from pyspark.sql.functions import col

sales_clean = (
    sales_clean
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("StockCode", "product_id")
    .withColumnRenamed("InvoiceNo", "transaction_id")
    .withColumnRenamed("InvoiceDate", "invoice_date")
    .withColumnRenamed("UnitPrice", "price")
)


In [0]:
from pyspark.sql.functions import broadcast

silver_sales_enriched = (
    sales_clean
    .join(broadcast(customer_clean), "customer_id", "left")
    .join(broadcast(products_clean), "product_id", "left")
).show()

In [0]:
silver_sales_enriched.printSchema()


In [0]:
silver_sales_enriched = silver_sales_enriched.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/retail_project/retail_db/retail_files/sliver/")

In [0]:
silver_final= silver_sales_enriched.select(
    "product_id",
    "customer_id",
    "transaction_id",
    "Quantity",
    "invoice_date",
    "price",
    "Country",
    "gender",
    "age",
    "country", 
    "product_name"    
  
)
